In [10]:
from printrun.printcore import printcore
import time
from subprocess import PIPE, Popen
import subprocess
import sys

### preconditions
usb port connect von windows aus
wsl usb port accept mit 2 befehlen 


GCode
G90: absolute koordinaten, G91: relative Koordinaten ab aktueller Position
M84: Motoren aus
G0 Y1: Gehe zu position y0, andere achsen nicht ändern
G28 Y: Home Y Achse (vgl. G0 Y0)

In [11]:
txt_location = "C:\\Users\\mah19\\OneDrive\\Desktop\\tmp.txt"
csv_location = "C:\\Users\\mah19\\OneDrive\\Desktop\\tmp.csv"
measure_duration = 12
pm_startup_duration = 3

In [12]:
def send_command(process, command):
    process.stdin.write(command)
    process.stdin.flush()
    process.stdin.write(b"\n")
    process.stdin.flush()

In [13]:
def init_proxmark():
    pm = subprocess.Popen(["wsl"], stdin=PIPE, stdout=PIPE)
    send_command(pm, b'cd ~/source/GITHUB/RfidResearchGroup/proxmark3 && pwd')
    send_command(pm, b'./pm3')
    time.sleep(pm_startup_duration)
    return pm

def run_hf_reader(command):
    pm = init_proxmark()
    print("measuring...")
    send_command(pm, command)
    print(command)
    
    stdout, _ = pm.communicate()
    stdout_str = stdout.decode('utf-8')
    lines = stdout_str.splitlines()
    for line in lines:
        if "successes" in line:
            print(line[4:7])
            return(line[4:7])
        
    return "000"

In [14]:
def init_printer():
    printer = printcore('COM7',250000)
    while not printer.online:
        time.sleep(0.1)
    return printer
  
def home(printer):
    printer.send_now("M84") #home bed
    input("Move bed to 0 and then press enter...")

def move(printer, i):
    printer.send_now('G1 Y' + str(i))
    time.sleep(1)
    print("now on " + str(i) + "mm")

def cleanup_printer(printer):  
    printer.disconnect()

# HF

In [ ]:
printer = init_printer()
home(printer)

In [17]:
start_distance = int(input("Ab wann interessanter Bereich:"))
end_distance = int(input("Bis wann interessanter Bereich:"))

print("Starting measurements...\r\n")

headers = [dst for dst in range(start_distance, end_distance + 1, 1)]
adpt = []
org = []
try:
    for i in headers:
        # move(printer, i)
        successes = run_hf_reader(b'hf 14a readerbaadpt')
        adpt.append(successes)
        successes = run_hf_reader(b'hf 14a readerbaorg')
        org.append(successes)
except KeyboardInterrupt:
    print('interrupted!')

print("adpt")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(adpt))
print("org")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(org))

Starting measurements...

measuring...
b'hf 14a readerbaadpt'
073
measuring...
b'hf 14a readerbaorg'
010
adpt
033
073
org
033
010


In [ ]:
print("Done, going to cleanup...")
cleanup_printer(printer)

# LF

In [ ]:
printer = init_printer()
home(printer)

In [ ]:
start_distance = int(input("Ab wann interessanter Bereich:"))
end_distance = int(input("Bis wann interessanter Bereich:"))

print("Starting measurements...\r\n")

headers = [dst for dst in range(start_distance, end_distance + 1, 1)]
adpt = []
org = []
try:
    for i in headers:
        # move(printer, i)
        successes = run_hf_reader(b'hf 14a readerbaadpt')
        adpt.append(successes)
        successes = run_hf_reader(b'hf 14a readerbaorg')
        org.append(successes)
except KeyboardInterrupt:
    print('interrupted!')

print("adpt")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(adpt))
print("org")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(org))

In [ ]:
print("Done, going to cleanup...")
cleanup_printer(printer)